In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.style.use("ggplot")

In [ ]:
PROPS = pd.read_csv("../output/validation/props.csv", index_col="run_id")
VARIATION_INDICES = [
    [0, 5], [5, 15], [15, 20]
]
PROPS

In [ ]:
frames = [pd.read_csv("../output/validation/dynamic_results.csv")]
for i in range(2, 1000):
    try:
        frames.append(pd.read_csv(f"../output/validation/dynamic_results_{i}.csv"))
    except FileNotFoundError:
        print(f"Stopping at {i}")
        break
SIM_DF_RAW = pd.concat(frames)

SIM_DFS_MEAN = [SIM_DF_RAW[(SIM_DF_RAW.run_id >= start) & (SIM_DF_RAW.run_id < end)].groupby("tick").mean() for start, end in VARIATION_INDICES]
SIM_DFS_STDDEV = [SIM_DF_RAW[(SIM_DF_RAW.run_id >= start) & (SIM_DF_RAW.run_id < end)].groupby("tick").std() for start, end in VARIATION_INDICES]
SIM_DFS_MEAN[0]

In [ ]:
PLOT_LIMITS = [
    {"start": 25, "end": 128},
]
PLOT_COLUMNS = [
    {
        "y.2": "total_white",
        "y.3": "total_purple",
        "y.4": "total_yellow",
        "y.5": "total_red",
        "y.6": "total_blue",
        "y.7": "total_orange",
        "y.8": "total_magenta",
        "y.9": "total_green",
    },
]

frames = []

for i in range(1000):
    try:
        inner_frames = []
        for j in range(1):
            df = pd.read_csv(f"../validation/run{i}.csv", skiprows=PLOT_LIMITS[j]["start"], nrows=PLOT_LIMITS[j]["end"] - PLOT_LIMITS[j]["start"] - 1)
            df = df.filter(["x", "y.2", "y.3", "y.4", "y.5", "y.6", "y.7", "y.8", "y.9"], axis=1)
            df = df.iloc[1:]
            df = df.convert_dtypes(convert_integer=True)
            df = df.rename({"x": "tick", **PLOT_COLUMNS[j]}, axis=1)
            df.tick -= 0.5
            df = df.set_index("tick")
            inner_frames.append(df)
        total_frame = pd.concat(inner_frames, axis=1, join="inner")
        total_frame["run_id"] = i
        frames.append(total_frame)
    except FileNotFoundError:
        print(f"Stopped at {i}")
        break

REAL_DF_RAW = pd.concat(frames)
REAL_DFS_MEAN = [REAL_DF_RAW[(REAL_DF_RAW.run_id >= start) & (REAL_DF_RAW.run_id < end)].groupby("tick").mean() for start, end in VARIATION_INDICES]
REAL_DFS_STDDEV = [REAL_DF_RAW[(REAL_DF_RAW.run_id >= start) & (REAL_DF_RAW.run_id < end)].groupby("tick").std() for start, end in VARIATION_INDICES]
REAL_DFS_MEAN[0]

In [ ]:
import matplotlib.pyplot as plt

categories = [
    ("total_blue", "blue", "blue"),
    ("total_orange", "orange", "orange"),
    ("total_red", "red", "red"),
    ("total_yellow", "yellow", "yellow"),
    ("total_green", "green", "green"),
    ("total_purple", "purple", "purple"),
    ("total_magenta", "magenta", "magenta"),
    ("total_white", "black", "white")
]

for i in range(len(VARIATION_INDICES)):
    fig, ax = plt.subplots(figsize=(16, 9))
    
    # RepastHPC Data
    for col, color, label_text in categories:
        ax.plot(SIM_DFS_MEAN[i][col], color=color, label=f"[RepastHPC] Total {label_text}")
        ax.fill_between(
            SIM_DFS_MEAN[i].index,
            SIM_DFS_MEAN[i][col] - SIM_DFS_STDDEV[i][col],
            SIM_DFS_MEAN[i][col] + SIM_DFS_STDDEV[i][col],
            color=color, alpha=0.2
        )
        
    # NetLogo Data
    for col, color, label_text in categories:
        ax.plot(REAL_DFS_MEAN[i][col], color=color, label=f"[NetLogo] Total {label_text}", marker=".", linewidth=0)
        ax.fill_between(
            REAL_DFS_MEAN[i].index,
            REAL_DFS_MEAN[i][col] - REAL_DFS_STDDEV[i][col],
            REAL_DFS_MEAN[i][col] + REAL_DFS_STDDEV[i][col],
            color=color, alpha=0.2, linestyle="--"
        )

    ax.set_title('Epistemic share over time')
    ax.set_xlabel('Tick')
    ax.set_ylabel('Agent count')
    ax.legend(bbox_to_anchor=(1, 1), loc='upper left')
    
    # plt.savefig(f"epistemic_share_variation_{i}.png", bbox_inches='tight')
    plt.show(fig)